# Task 1: Data Exploration and Preprocessing

Load the GSE50081 NSCLC gene expression dataset from Lab 7, use clinical metadata to classify **Adenocarcinoma vs Squamous Cell Carcinoma**, apply feature selection, and split into training/testing subsets.

**Dataset**: GSE50081 (Non-Small Cell Lung Cancer, 169 samples after filtering)
**Classification Task**: Adenocarcinoma (1) vs Squamous Cell Carcinoma (0)

## 1. Initialize Project Environment

In [1]:
import logging
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Python {sys.version}")
print(f"numpy {np.__version__}")
print(f"pandas {pd.__version__}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]
numpy 2.1.3
pandas 2.2.3


## 2. Define Configuration Parameters

In [2]:
@dataclass
class TaskConfig:
    handle: str
    export_dir: Path = Path("artifacts")
    data_dir: Path = Path("data")
    n_features: int = 100  # Select top N most informative genes
    test_size: float = 0.2
    random_state: int = 42

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        info["data_dir"] = str(info["data_dir"])
        return info


CONFIG = TaskConfig(handle="rbals")
CONFIG.describe()

{'handle': 'rbals',
 'export_dir': 'artifacts',
 'data_dir': 'data',
 'n_features': 100,
 'test_size': 0.2,
 'random_state': 42}

## 3. Implement Core Functionality

In [3]:
def load_and_preprocess_data(config: TaskConfig):
    # Load Lab 7 preprocessed expression data
    expr_path = config.data_dir / "task1_preprocessed_expression.csv"
    expr_df = pd.read_csv(expr_path)

    # Transpose: samples as rows, genes as columns
    expr_df = expr_df.set_index("Gene").T
    expr_df.index.name = "SampleID"
    expr_df = expr_df.reset_index()

    logging.info(
        f"Expression data: {expr_df.shape[0]} samples, {expr_df.shape[1] - 1} genes"
    )

    # Load real GSE50081 clinical metadata
    metadata_path = config.data_dir / "GSE50081_metadata.csv"
    metadata = pd.read_csv(metadata_path)

    # Filter to only Adenocarcinoma and Squamous Cell Carcinoma
    valid_histology = ["adenocarcinoma", "squamous cell carcinoma"]
    metadata = metadata[metadata["histology"].isin(valid_histology)].copy()

    logging.info(f"Filtered to {len(metadata)} samples (Adeno + Squamous)")

    # Merge expression with metadata
    df = expr_df.merge(metadata[["SampleID", "histology"]], on="SampleID")

    # Convert labels: Adenocarcinoma=1, Squamous=0
    df["target"] = (df["histology"] == "adenocarcinoma").astype(int)

    logging.info(f"Dataset dimensions: {df.shape}")
    logging.info(f"Target distribution:\\n{df['target'].value_counts()}")
    logging.info(f"  1 = Adenocarcinoma, 0 = Squamous Cell Carcinoma")

    # Check for missing values
    missing_values = df.isnull().sum().sum()
    logging.info(f"Missing values: {missing_values}")

    # Split into features (X) and target (y)
    feature_cols = [
        c for c in df.columns if c not in ["SampleID", "histology", "target"]
    ]
    X = df[feature_cols]
    y = df["target"]

    # Split into training and testing subsets FIRST (before feature selection)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.test_size, random_state=config.random_state, stratify=y
    )

    # Feature selection: Select top N most informative genes using ANOVA F-test
    selector = SelectKBest(f_classif, k=config.n_features)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_test_selected = selector.transform(X_test)

    # Get selected feature names
    selected_mask = selector.get_support()
    selected_features = [f for f, s in zip(feature_cols, selected_mask) if s]
    logging.info(f"Selected {len(selected_features)} top features via ANOVA F-test")

    # Normalization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_selected)
    X_test_scaled = scaler.transform(X_test_selected)

    # Convert back to DataFrame
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=selected_features)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=selected_features)

    # Save raw merged data (with selected features only)
    df_raw = df[["target"] + selected_features].copy()

    return (
        df_raw,
        X_train_scaled_df,
        X_test_scaled_df,
        y_train.reset_index(drop=True),
        y_test.reset_index(drop=True),
        selected_features,
    )


df_raw, X_train, X_test, y_train, y_test, selected_genes = load_and_preprocess_data(
    CONFIG
)

16:29:18 | INFO | Expression data: 181 samples, 2001 genes
16:29:18 | INFO | Filtered to 169 samples (Adeno + Squamous)
16:29:18 | INFO | Dataset dimensions: (169, 2004)
16:29:18 | INFO | Target distribution:\ntarget
1    127
0     42
Name: count, dtype: int64
16:29:18 | INFO |   1 = Adenocarcinoma, 0 = Squamous Cell Carcinoma
16:29:18 | INFO | Missing values: 0
16:29:18 | INFO | Selected 100 top features via ANOVA F-test


## 4. Validate with Unit Tests

In [4]:
# Assertions to ensure data integrity
assert df_raw.shape[0] == 169, (
    f"Dataset size mismatch: expected 169, got {df_raw.shape[0]}"
)
assert df_raw.shape[1] == CONFIG.n_features + 1, (
    f"Expected {CONFIG.n_features + 1} columns"
)
assert not df_raw.isnull().values.any(), "Missing values found"
assert len(X_train) + len(X_test) == 169, "Split mismatch"
print(f"[OK] Validation passed: {len(selected_genes)} features selected")

[OK] Validation passed: 100 features selected


## 5. Export Results

In [5]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save the raw dataset
raw_out = EXPORT_DIR / "task1_cancer_gene_expression.csv"
df_raw.to_csv(raw_out, index=False)
print(f"[OK] Raw dataset saved to: {raw_out.resolve()}")

# Save preprocessed splits
X_train.to_csv(EXPORT_DIR / "task1_X_train_scaled.csv", index=False)
X_test.to_csv(EXPORT_DIR / "task1_X_test_scaled.csv", index=False)
y_train.to_csv(EXPORT_DIR / "task1_y_train.csv", index=False)
y_test.to_csv(EXPORT_DIR / "task1_y_test.csv", index=False)

# Save selected gene names
genes_out = EXPORT_DIR / "task1_selected_genes.txt"
with open(genes_out, "w") as f:
    f.write("\n".join(selected_genes))
print(f"[OK] Selected genes list saved to: {genes_out}")
print(f"[OK] Preprocessed splits saved to artifacts/")

[OK] Raw dataset saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/08_ML_flower/assignments/artifacts/task1_cancer_gene_expression.csv
[OK] Selected genes list saved to: artifacts/task1_selected_genes.txt
[OK] Preprocessed splits saved to artifacts/
